# Data processing - shots

In [1]:
import pandas as pd

DATA_PATH = "../../data/"

In [2]:
def get_season(date):
    year = date.year
    return year if date.month >= 9 else year - 1

## Load shot detail data

In [3]:
import glob

shot_files = sorted(glob.glob(DATA_PATH + "/nba_play_by_play_shot_data/shotdetail*.csv"))

len(shot_files)

58

## Process data

In [ ]:
# GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,
# EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM

# hoop is at (LOC_X, LOC_Y) = (0, 0)

cols_to_keep = [
    "GAME_DATE", "PLAYER_ID",
    "SHOT_TYPE", "LOC_X", "LOC_Y",
    "SHOT_ATTEMPTED_FLAG", "SHOT_MADE_FLAG"
]

frames = []

for file_path in shot_files:
    is_playoffs = "_po_" in file_path.lower()

    df = pd.read_csv(file_path, usecols=lambda c: c in cols_to_keep)

    # 1 = actual shot attempt
    if "SHOT_ATTEMPTED_FLAG" in df.columns:
        df = df[df["SHOT_ATTEMPTED_FLAG"] == 1]

    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y%m%d", errors="coerce")
    df = df[df["GAME_DATE"].notna()]
    df["season"] = df["GAME_DATE"].apply(get_season)
    df = df.drop(columns=["GAME_DATE", "SHOT_ATTEMPTED_FLAG"])

    # cleanup coordinates
    df["LOC_X"] = pd.to_numeric(df["LOC_X"], errors="coerce")
    df["LOC_Y"] = pd.to_numeric(df["LOC_Y"], errors="coerce")
    df = df.dropna(subset=["LOC_X", "LOC_Y"])

    df["playoffs"] = 1 if is_playoffs else 0

    frames.append(df)

shot_events = pd.concat(frames, ignore_index=True)

shot_events = shot_events.rename(columns={
    "PLAYER_ID": "personId",
    "SHOT_TYPE": "shotType",
    "LOC_X": "locX",
    "LOC_Y": "locY",
    "SHOT_MADE_FLAG": "shotMade",
})

# SHOT_TYPE: 3PT Field Goal or 2PT Field Goal
shot_events["shotValue"] = shot_events["shotType"].str.extract(r"(\d)PT")[0].astype("Int64")

shot_events = shot_events[[
    "season", "personId", "shotValue", "locX", "locY", "shotMade", "playoffs"
]]

shot_events.head()

### Check range for locX, locY

In [5]:
print("locX:", (shot_events["locX"].min(), shot_events["locX"].max()))
print("locY:", (shot_events["locY"].min(), shot_events["locY"].max()))

locX: (np.int64(-250), np.int64(250))
locY: (np.int64(-52), np.int64(884))


## Save data

In [ ]:
from pathlib import Path

output_dir = Path(DATA_PATH) / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

ranges = [
    ("shot_events_1996_2006.csv", 1996, 2006),
    ("shot_events_2007_2015.csv", 2007, 2015),
    ("shot_events_2016_2024.csv", 2016, 2024),
]

for filename, start, end in ranges:
    part = shot_events[(shot_events["season"] >= start) & (shot_events["season"] <= end)]
    part.to_csv(output_dir / filename, index=False)
    print(f"{filename}: {len(part):,} rows")